In [1]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Feature storage
# ---------------------------------------------------------------------------
feat_headers = [
    'window_id', 'time_ms',
    'avg_velocity', 'avg_velocity_off',
    'avg_ioi', 'avg_duration', 'avg_articulation',
    'pitch_range', 'avg_pitch', 'avg_pitch_step',
    'num_chords', 'avg_notes_per_chord', 'chord_density',
    'avg_polyphony',
    'avg_arpeggiation_speed', 'avg_arpeggation_distance',
    'num_grace_notes', 'avg_grace_duration', 'avg_grace_interval'
]

# ---------------------------------------------------------------------------
# Arpeggio detection
# ---------------------------------------------------------------------------
def FEAT_arpeggio(df_win, time_window=500, max_interval=10, min_notes=3):
    n = len(df_win)
    df_win = df_win.reset_index(drop=True)

    sequences = []

    def dfs(path):
        last = path[-1]
        extended = False

        for j in range(last + 1, n):
            dt = df_win.iloc[j]["time_ms"] - df_win.iloc[last]["time_ms"]
            if dt > time_window:
                break

            interval = df_win.iloc[j]["note"] - df_win.iloc[last]["note"]

            if dt == 0:
                continue

            effectively_simultaneous = dt <= GRACE_MS
            ascending = interval > 0 if not effectively_simultaneous else interval != 0
            overlaps = df_win.iloc[last]["time_off_ms"] > df_win.iloc[j]["time_ms"]

            if ascending and abs(interval) <= max_interval and overlaps:
                dfs(path + [j])
                extended = True

        if not extended and len(path) >= min_notes:
            sequences.append(path)

    for i in range(n):
        dfs([i])

    return sequences

# ---------------------------------------------------------------------------
# Grace note detection
# ---------------------------------------------------------------------------
def FEAT_grace_notes(df_win, max_dt=150, max_semitone=2, max_duration=150):
    df_win = df_win.sort_values("time_ms").reset_index(drop=True)
    n = len(df_win)

    count = 0
    durations = []
    intervals = []

    for i in range(n - 1):
        note_i = df_win.iloc[i]
        note_j = df_win.iloc[i + 1]

        dt = note_j["time_ms"] - note_i["time_ms"]
        interval = abs(note_j["note"] - note_i["note"])
        duration = note_i["duration_ms"]

        if (
            0 < dt <= max_dt and
            interval <= max_semitone and
            duration <= max_duration
        ):
            count += 1
            durations.append(duration)
            intervals.append(interval)

    if count == 0:
        return 0, np.nan, np.nan

    return count, np.mean(durations), np.mean(intervals)

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def compute_ioi(times_ms):
    times = np.sort(times_ms)
    if len(times) < 2:
        return np.array([np.nan])
    return np.diff(times)

def compute_articulation(durations_ms, ioi_values):
    n = min(len(durations_ms), len(ioi_values))
    if n == 0:
        return np.nan

    ratios = np.array(durations_ms[:n]) / np.where(ioi_values[:n] == 0, np.nan, ioi_values[:n])
    return np.nanmean(ratios)

# ---------------------------------------------------------------------------
# Arpeggio features
# ---------------------------------------------------------------------------
def arpeggio_features(df_win, time_window=500, min_notes=3):
    df_win = df_win.reset_index(drop=True)

    used = set()
    num_chords = 0
    all_speeds = []
    all_distances = []
    notes_per_chord = []

    seqs = FEAT_arpeggio(df_win, time_window=time_window, min_notes=min_notes)

    for seq in seqs:
        if any(i in used for i in seq):
            continue

        for i in seq:
            used.add(i)

        num_chords += 1
        notes_per_chord.append(len(seq))

        arp_notes = df_win.iloc[seq]
        all_speeds.extend(np.diff(arp_notes["time_ms"].values))
        all_distances.extend(np.diff(arp_notes["note"].values))

    if num_chords == 0:
        return 0, np.nan, np.nan, np.nan

    return (
        num_chords,
        np.mean(notes_per_chord),
        np.mean(all_speeds) if all_speeds else np.nan,
        np.mean(all_distances) if all_distances else np.nan
    )

# ---------------------------------------------------------------------------
# Polyphony
# ---------------------------------------------------------------------------
def compute_polyphony(df_win):
    active_counts = []
    times = df_win["time_ms"].values

    for t in times:
        active = np.sum(
            (df_win["time_ms"] <= t) &
            (df_win["time_off_ms"] > t)
        )
        active_counts.append(active)

    return np.mean(active_counts)

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def compute_ioi(times_ms):
    times = np.sort(times_ms)
    if len(times) < 2:
        return np.array([np.nan])
    return np.diff(times)

def compute_articulation(durations_ms, ioi_values):
    n = min(len(durations_ms), len(ioi_values))
    if n == 0:
        return np.nan

    ratios = np.array(durations_ms[:n]) / np.where(ioi_values[:n] == 0, np.nan, ioi_values[:n])
    return np.nanmean(ratios)

# ---------------------------------------------------------------------------
# Arpeggio features
# ---------------------------------------------------------------------------
def arpeggio_features(df_win, time_window=500, min_notes=3):
    df_win = df_win.reset_index(drop=True)

    used = set()
    num_chords = 0
    all_speeds = []
    all_distances = []
    notes_per_chord = []

    seqs = FEAT_arpeggio(df_win, time_window=time_window, min_notes=min_notes)

    for seq in seqs:
        if any(i in used for i in seq):
            continue

        for i in seq:
            used.add(i)

        num_chords += 1
        notes_per_chord.append(len(seq))

        arp_notes = df_win.iloc[seq]
        all_speeds.extend(np.diff(arp_notes["time_ms"].values))
        all_distances.extend(np.diff(arp_notes["note"].values))

    if num_chords == 0:
        return 0, np.nan, np.nan, np.nan

    return (
        num_chords,
        np.mean(notes_per_chord),
        np.mean(all_speeds) if all_speeds else np.nan,
        np.mean(all_distances) if all_distances else np.nan
    )

# ---------------------------------------------------------------------------
# Polyphony
# ---------------------------------------------------------------------------
def compute_polyphony(df_win):
    active_counts = []
    times = df_win["time_ms"].values

    for t in times:
        active = np.sum(
            (df_win["time_ms"] <= t) &
            (df_win["time_off_ms"] > t)
        )
        active_counts.append(active)

    return np.mean(active_counts)

# ---------------------------------------------------------------------------
# Main loop
# ---------------------------------------------------------------------------

for n in range(4):
	
	df = pd.read_csv("recorded_piano"+str(n)+".csv")
	df = df.sort_values("time_ms").reset_index(drop=True)

	
	
	WINDOW_MS = 15000
	GRACE_MS = 10

	start_time = df["time_ms"].min()
	end_time = df["time_ms"].max()

	window_id = 0
	rows = []

	t = start_time

	while t < end_time:
		t_end = t + WINDOW_MS
		mask = (df["time_ms"] >= t) & (df["time_ms"] < t_end)
		df_win = df[mask].copy()

		if len(df_win) == 0:
			t = t_end
			window_id += 1
			continue

		avg_velocity = df_win["velocity_on"].mean()
		avg_velocity_off = df_win["velocity_off"].mean() if "velocity_off" in df_win else np.nan

		avg_duration = df_win["duration_ms"].mean()

		ioi_vals = compute_ioi(df_win["time_ms"].values)
		avg_ioi = np.nanmean(ioi_vals)

		avg_articulation = compute_articulation(
			df_win["duration_ms"].values,
			ioi_vals
		)

		# Pitch features
		pitch_range = df_win["note"].max() - df_win["note"].min()
		avg_pitch = df_win["note"].mean()
		diffs = np.diff(df_win["note"].values)
		avg_pitch_step = np.mean(np.abs(diffs)) if len(diffs) else np.nan

		# Arpeggios
		num_chords, avg_notes_per_chord, avg_arp_speed, avg_arp_distance = arpeggio_features(df_win)

		# Density
		chord_density = num_chords / (WINDOW_MS / 1000)

		# Polyphony
		avg_polyphony = compute_polyphony(df_win)

		# Grace notes
		num_grace, avg_grace_dur, avg_grace_int = FEAT_grace_notes(df_win)

		rows.append({
			'window_id': window_id,
			'time_ms': t,
			'avg_velocity': round(avg_velocity, 3),
			'avg_velocity_off': round(avg_velocity_off, 3) if not np.isnan(avg_velocity_off) else np.nan,
			'avg_ioi': round(avg_ioi, 3),
			'avg_duration': round(avg_duration, 3),
			'avg_articulation': round(avg_articulation, 3),
			'pitch_range': pitch_range,
			'avg_pitch': round(avg_pitch, 3),
			'avg_pitch_step': round(avg_pitch_step, 3) if not np.isnan(avg_pitch_step) else np.nan,
			'num_chords': num_chords,
			'avg_notes_per_chord': round(avg_notes_per_chord, 3) if not np.isnan(avg_notes_per_chord) else np.nan,
			'chord_density': round(chord_density, 3),
			'avg_polyphony': round(avg_polyphony, 3),
			'avg_arpeggiation_speed': round(avg_arp_speed, 3) if not np.isnan(avg_arp_speed) else np.nan,
			'avg_arpeggation_distance': round(avg_arp_distance, 3) if not np.isnan(avg_arp_distance) else np.nan,
			'num_grace_notes': num_grace,
			'avg_grace_duration': round(avg_grace_dur, 3) if not np.isnan(avg_grace_dur) else np.nan,
			'avg_grace_interval': round(avg_grace_int, 3) if not np.isnan(avg_grace_int) else np.nan
		})

		t = t_end
		window_id += 1

	# ---------------------------------------------------------------------------
	# Save output
	# ---------------------------------------------------------------------------
	df_features = pd.DataFrame(rows, columns=feat_headers)
	df_features.to_csv("features"+str(n)+".csv", index=False)

	print("\nExtracted features:")
	print(df_features.to_string(index=False))
	print(f"\n{len(df_features)} windows written to features.csv")


Extracted features:
 window_id  time_ms  avg_velocity  avg_velocity_off  avg_ioi  avg_duration  avg_articulation  pitch_range  avg_pitch  avg_pitch_step  num_chords  avg_notes_per_chord  chord_density  avg_polyphony  avg_arpeggiation_speed  avg_arpeggation_distance  num_grace_notes  avg_grace_duration  avg_grace_interval
         0        0        47.000            32.101  218.294       707.145            15.292           34     73.565           7.279          11                3.636          0.733          3.739                 136.655                     2.069                1               129.0                 2.0
         1    15000        51.685            33.493  193.639       654.644            17.082           34     72.795           8.361           7                4.000          0.467          3.521                 200.476                     2.857                0                 NaN                 NaN
         2    30000        58.500            35.333  155.208       406

FileNotFoundError: [Errno 2] No such file or directory: 'recorded_piano2.csv'